# 셀 데이터 전처리
- 이 파일은 데이터 분석을 위한 전처리를 수행합니다.
- 50개의 충/방전 셀 데이터를 일괄적으로 처리할 수 있도록 하였습니다.

<br/>

## 과정
1. 원본 데이터 불러오기 및 데이터프레임 구성
2. 전처리 전 데이터 품질지수 산출
3. 특성 선택
4. 결측치 처리
4. 이상치 처리
5. 데이터 변환
6. 전처리 후 데이터 품질지수 산출 및 비교


In [ ]:
# 라이브러리
print("===== 라이브러리 import 실행 ======")
try:
    import pandas as pd
    import os
    print("라이브러리 import 완료")
except Exception as e:
    print(f"오류 발생: {e}")
print("")


===== 라이브러리 import 실행 ======
라이브러리 import 완료



## 1. 원본 데이터 불러오기 및 데이터프레임 구성

In [6]:
try:
    # 입출력 폴더 경로 설정
    input_dir = '../data/raw'
    bms_output_dir = '../data/processed/raw_bms'
    cell_output_dir = '../data/processed/raw_cell'
    os.makedirs(bms_output_dir, exist_ok=True)
    os.makedirs(cell_output_dir, exist_ok=True)

    # 충/방전 데이터 선택
    # 충전: chg, 방전: dchg
    mode = 'dchg'

    # 모든 CSV 파일 순회
    print(f"{mode} 파일 접근 중")
    for filename in os.listdir(input_dir):
        if filename.endswith(f'_{mode}.csv'):
            filepath = os.path.join(input_dir, filename)
            bms_output_path = os.path.join(bms_output_dir, f'bms_{filename}')
            cell_output_path = os.path.join(cell_output_dir, f'cell_{filename}')
            df = pd.read_csv(filepath)

            # 열 필터링 조건 적용
            df = pd.read_csv(filepath)
            bms_cols = [col for col in df.columns
                        if not col.startswith('M') # 제외 조건
                        ]   
            cell_cols = [col for col in df.columns
                         if col.startswith('M') or ('Date' in col or 'Time' in col or 'SerialNumber' in col)
                         ]
            
            df_bms = df[bms_cols]
            df_cell = df[cell_cols]

            # 모두 NaN인 행 제거
            df_bms = df_bms.dropna(how='all')
            df_cell = df_cell.dropna(how='all')

            # 새로운 이름으로 저장
            print(f"====== {filename} ======")
            bms_output_path = os.path.join(bms_output_dir, f'bms_{filename}')
            df_bms.to_csv(bms_output_path, index=False)
            print(f"bms 데이터 처리 완료 => {bms_output_path}")

            cell_output_path = os.path.join(cell_output_dir, f'cell_{filename}')
            df_cell.to_csv(cell_output_path, index=False)
            print(f"cell 데이터 처리 완료 => {cell_output_path}")
            print("=========================")
            print("")
    print("데이터 분리 완료")

except Exception as e:
    print(f"오류 발생: {e}")

dchg 파일 접근 중
====== 1000_dchg.csv ======
bms 데이터 처리 완료 => ../data/processed/raw_bms\bms_1000_dchg.csv
cell 데이터 처리 완료 => ../data/processed/raw_cell\cell_1000_dchg.csv

====== 1001_dchg.csv ======
bms 데이터 처리 완료 => ../data/processed/raw_bms\bms_1001_dchg.csv
cell 데이터 처리 완료 => ../data/processed/raw_cell\cell_1001_dchg.csv

====== 1002_dchg.csv ======
bms 데이터 처리 완료 => ../data/processed/raw_bms\bms_1002_dchg.csv
cell 데이터 처리 완료 => ../data/processed/raw_cell\cell_1002_dchg.csv

====== 1003_dchg.csv ======
bms 데이터 처리 완료 => ../data/processed/raw_bms\bms_1003_dchg.csv
cell 데이터 처리 완료 => ../data/processed/raw_cell\cell_1003_dchg.csv

====== 1004_dchg.csv ======
bms 데이터 처리 완료 => ../data/processed/raw_bms\bms_1004_dchg.csv
cell 데이터 처리 완료 => ../data/processed/raw_cell\cell_1004_dchg.csv

====== 1005_dchg.csv ======
bms 데이터 처리 완료 => ../data/processed/raw_bms\bms_1005_dchg.csv
cell 데이터 처리 완료 => ../data/processed/raw_cell\cell_1005_dchg.csv

====== 1006_dchg.csv ======
bms 데이터 처리 완료 => ../data/processed/

## 2. 전처리 전 데이터 품질지수 산출

In [7]:
# 품질지수 산출
print("품질 지수 산출을 시작합니다.")
try:
    # 입력 폴더 경로 설정
    input_dir = '../data/processed/raw_cell'

    # 폴더 내 파일 순회
    print("파일 접근 중")
    isnull_list = []
    for filename in os.listdir(input_dir):
        filepath = os.path.join(input_dir, filename)
        df = pd.read_csv(filepath)
        print(f"====== {filename} ======")
        # 2-1. 완전성 품질지수

        print(f"완전성 품질지수: {100*(1-(sum(df.isnull().sum())/df.size))}")

        # 2-2. 유일성 품질지수
        uniqueness_score = 100*df['Time'].nunique()/len(df)
        print(f"유일성 품질지수: {uniqueness_score}")
        
        # 2-3. 유효성 품질지수
        pattern = r'^M(0[1-9]|1[0-6])CV(0[1-9]|1[0-1])$'
        validity_score = 100*(df.size-(df.loc[:, df.columns.str.match(pattern)] < 0).sum().sum())/df.size
        print(f"유효성 품질지수: {validity_score}")

        # 2-4. 일관성 품질 지수
        # 일관성 규칙: 적용할 규칙 없음
        consistency_score = None
        print("일관성 품질지수: 평가 불가")

        # 2-5. 정확성 품질 지수
        # 정확성 조건: 없음
        print("정확성 품질지수: 평가 불가")

        # 2-6. 무결성 품질 지수
        scores = [uniqueness_score, validity_score, consistency_score]
        count = 0
        for score in scores:
            if score is not None and score != 100:
                count += 1
        integrity_score = 100*(1-(count)/len(scores))
        print(f"무결성 품질지수: {integrity_score}")
        print("=========================")
        print("")
    print("품질지수 산출 완료")
except Exception as e:
    print(f"오류 발생: {e}")

품질 지수 산출을 시작합니다.
파일 접근 중
====== cell_1000_chg.csv ======
완전성 품질지수: 100.0
유일성 품질지수: 100.0
유효성 품질지수: 100.0
일관성 품질지수: 평가 불가
정확성 품질지수: 평가 불가
무결성 품질지수: 100.0

====== cell_1000_dchg.csv ======
완전성 품질지수: 100.0
유일성 품질지수: 100.0
유효성 품질지수: 100.0
일관성 품질지수: 평가 불가
정확성 품질지수: 평가 불가
무결성 품질지수: 100.0

====== cell_1001_chg.csv ======
완전성 품질지수: 100.0
유일성 품질지수: 100.0
유효성 품질지수: 100.0
일관성 품질지수: 평가 불가
정확성 품질지수: 평가 불가
무결성 품질지수: 100.0

====== cell_1001_dchg.csv ======
완전성 품질지수: 100.0
유일성 품질지수: 100.0
유효성 품질지수: 100.0
일관성 품질지수: 평가 불가
정확성 품질지수: 평가 불가
무결성 품질지수: 100.0

====== cell_1002_chg.csv ======
완전성 품질지수: 100.0
유일성 품질지수: 100.0
유효성 품질지수: 100.0
일관성 품질지수: 평가 불가
정확성 품질지수: 평가 불가
무결성 품질지수: 100.0

====== cell_1002_dchg.csv ======
완전성 품질지수: 100.0
유일성 품질지수: 100.0
유효성 품질지수: 100.0
일관성 품질지수: 평가 불가
정확성 품질지수: 평가 불가
무결성 품질지수: 100.0

====== cell_1003_chg.csv ======
완전성 품질지수: 100.0
유일성 품질지수: 100.0
유효성 품질지수: 100.0
일관성 품질지수: 평가 불가
정확성 품질지수: 평가 불가
무결성 품질지수: 100.0

====== cell_1003_dchg.csv ======
완전성 품질지수: 100.0
유일성 품질지수: 100.0
유효성 

## 3. 특성 선택: 
- 1. 0에 가까운 분산을 가진 특성 제거
- 2. 상관 분석을 통한 다중공선성 확인 및 특성 제거
    - 다중공선성은 현재 분석 목표에서 무시해도 되는 특징이기 때문에 진행하지 않음

### 3-1. 0에 가까운 분산을 가진 특성 제거
- |std| < 0.005 인 변수가 존재하지 않기 때문에 실행할 필요 없음

In [8]:
# 0에 가까운 분산을 가진 특성 제거
## |std| < 0.005 인 변수 제거
print("분산값 기반한 특성 제거를 시작합니다.")
try:
    # 입력 폴더 경로 설정
    input_dir = '../data/processed/raw_cell'

    # 폴더 내 파일 순회
    print("파일 접근 중")
    isnull_list = []
    for filename in os.listdir(input_dir):
        filepath = os.path.join(input_dir, filename)
        print(f"====== {filename} ======")
        print("|std| < 0.005 인 특성을 찾는 중.")
        features =df.drop(columns=['Date', 'Time', 'SerialNumber'])
        df_stat = features.describe().transpose()
        df_std_0 = df_stat[df_stat['std'] < 0.005]

        if df_std_0.shape[0] >= 1:
            print("조건을 만족하는 특성이 존재합니다.")
            print(f"제거 대상 특성: {list(df_std_0.index)}")
            df_reduced = features.drop(columns=df_std_0.index)
        else:
            print("조건을 만족하는 특성이 없습니다.")
            df_reduced = features.copy()
        print("===============================")
        print("")
    print("특성 제거 종료")
except Exception as e:
    print(f"오류 발생: {e}")

분산값 기반한 특성 제거를 시작합니다.
파일 접근 중
====== cell_1000_chg.csv ======
|std| < 0.005 인 특성을 찾는 중.
조건을 만족하는 특성이 없습니다.

====== cell_1000_dchg.csv ======
|std| < 0.005 인 특성을 찾는 중.
조건을 만족하는 특성이 없습니다.

====== cell_1001_chg.csv ======
|std| < 0.005 인 특성을 찾는 중.
조건을 만족하는 특성이 없습니다.

====== cell_1001_dchg.csv ======
|std| < 0.005 인 특성을 찾는 중.
조건을 만족하는 특성이 없습니다.

====== cell_1002_chg.csv ======
|std| < 0.005 인 특성을 찾는 중.
조건을 만족하는 특성이 없습니다.

====== cell_1002_dchg.csv ======
|std| < 0.005 인 특성을 찾는 중.
조건을 만족하는 특성이 없습니다.

====== cell_1003_chg.csv ======
|std| < 0.005 인 특성을 찾는 중.
조건을 만족하는 특성이 없습니다.

====== cell_1003_dchg.csv ======
|std| < 0.005 인 특성을 찾는 중.
조건을 만족하는 특성이 없습니다.

====== cell_1004_chg.csv ======
|std| < 0.005 인 특성을 찾는 중.
조건을 만족하는 특성이 없습니다.

====== cell_1004_dchg.csv ======
|std| < 0.005 인 특성을 찾는 중.
조건을 만족하는 특성이 없습니다.

====== cell_1005_chg.csv ======
|std| < 0.005 인 특성을 찾는 중.
조건을 만족하는 특성이 없습니다.

====== cell_1005_dchg.csv ======
|std| < 0.005 인 특성을 찾는 중.
조건을 만족하는 특성이 없습니다.

====== cell_1006_chg.csv ===

## 4. 결측치 처리

In [9]:
print("결측치 처리를 시작합니다.")
try:
    # 입력 폴더 경로 설정
    input_dir = '../data/processed/raw_cell'

    # 폴더 내 파일 순회
    print("파일 접근 중")
    isnull_list = []
    for filename in os.listdir(input_dir):
        filepath = os.path.join(input_dir, filename)
        print(f"====== {filename} ======")
        df = pd.read_csv(filepath)
        print(f"결측치 개수: {sum(df.isnull().sum())}")
        if sum(df.isnull().sum()) >= 1:
            print("결측치 제거 수행")
            df_cleaned = df.dropna(how='any', subset=[col for col in df.columns if col not in ['Date', 'Time', 'SerialNumber']])
            print(f"레코드 수: {df.shape[0]-df_cleaned.shape[0]}개 삭제 => {df_cleaned.shape[0]}개")
        print("===============================")
        print("")
except Exception as e:
    print(f"오류 발생: {e}")

결측치 처리를 시작합니다.
파일 접근 중
====== cell_1000_chg.csv ======
결측치 개수: 0

====== cell_1000_dchg.csv ======
결측치 개수: 0

====== cell_1001_chg.csv ======
결측치 개수: 0

====== cell_1001_dchg.csv ======
결측치 개수: 0

====== cell_1002_chg.csv ======
결측치 개수: 0

====== cell_1002_dchg.csv ======
결측치 개수: 0

====== cell_1003_chg.csv ======
결측치 개수: 0

====== cell_1003_dchg.csv ======
결측치 개수: 0

====== cell_1004_chg.csv ======
결측치 개수: 0

====== cell_1004_dchg.csv ======
결측치 개수: 0

====== cell_1005_chg.csv ======
결측치 개수: 0

====== cell_1005_dchg.csv ======
결측치 개수: 0

====== cell_1006_chg.csv ======
결측치 개수: 0

====== cell_1006_dchg.csv ======
결측치 개수: 0

====== cell_1007_chg.csv ======
결측치 개수: 0

====== cell_1007_dchg.csv ======
결측치 개수: 0

====== cell_1008_chg.csv ======
결측치 개수: 0

====== cell_1008_dchg.csv ======
결측치 개수: 0

====== cell_1009_chg.csv ======
결측치 개수: 0

====== cell_1009_dchg.csv ======
결측치 개수: 0

====== cell_1010_chg.csv ======
결측치 개수: 0

====== cell_1010_dchg.csv ======
결측치 개수: 0

====== cell_1011_chg

## 5. 이상치 처리
- 분석 목표에 기반하여, 이상치를 주요 정보로 판단하고 처리하지 않도록 합니다.

## 6. 데이터 변환
- 분석 목표를 위해 변수의 스케일을 변환하지 않습니다.

## 7. 전처리 후 품질지수 산출 및 비교
- 전처리 이전 모든 데이터의 품질지수가 100 이므로 진행하지 않습니다.